In [ ]:
import pandas as pd
import unicodedata

In [ ]:
import sys
print(sys.executable)

In [ ]:
#Antes foi necessário rodar o import sys !{sys.executable} -m pip install pyarrow fastparquet
import pyarrow
import fastparquet

In [ ]:
df_bronze = pd.read_csv("../data/1-bronze/bronze_acidentes.csv", encoding="utf-8", low_memory=False
)

df_silver = df_bronze.copy()

In [ ]:
print(list(df_silver.columns))

In [ ]:
df_silver.isnull().sum()[df_silver.isnull().sum() > 0]

### Tratando nulos

In [ ]:
#Colunas com categórias

colunas_categoricas = ['id_veiculo', 'tipo_veiculo', 'marca', 'tipo_envolvido', 'estado_fisico', 'idade', 'sexo', 'nacionalidade', 'naturalidade', 'regional', 'delegacia', 'uop']

for col in colunas_categoricas:
    df_silver[col] = df_silver[col].fillna("desconhecido")

df_silver.isnull().sum()[df_silver.isnull().sum() > 0]

In [ ]:
#Colunas númericas direcionada para vitimas

coluna_vitimas = ['ilesos', 'feridos_leves', 'feridos_graves', 'mortos']
for col in coluna_vitimas:
    df_silver[col] = df_silver[col].fillna(0)

df_silver.isnull().sum()[df_silver.isnull().sum() > 0]

In [ ]:
# Exclusão de colunas que os dados não serão utilizados neste projeto.
df_silver = df_silver.drop(columns=['latitude', "longitude", 'arquivo_origem'])
# Excluindo linhas que não possuem a data
total_linhas_original = len(df_silver)
df_silver = df_silver.dropna(subset=['data_inversa'])

### Conversão de tipos de dados

In [ ]:
colunas_int = [
    'ano_fabricacao_veiculo',
    'idade',
    'ilesos',
    'feridos_leves',
    'feridos_graves',
    'mortos'
]

df_silver[colunas_int] = df_silver[colunas_int].apply(
    lambda x: pd.to_numeric(x, errors='coerce').astype('Int64')
)

In [ ]:
colunas_string = [
    'dia_semana',
    'uf',
    'municipio',
    'causa_acidente',
    'tipo_acidente',
    'classificacao_acidente',
    'fase_dia',
    'condicao_metereologica',
    'tipo_veiculo',
    'estado_fisico'
]

df_silver[colunas_string] = df_silver[colunas_string].astype('string')

In [ ]:
df_silver["horario"] = pd.to_datetime(
    df_silver["horario"], errors="coerce"
)
df_silver['horario'] = df_silver['horario'].dt.hour
df_silver['horario'].head()

In [ ]:
df_silver['data_inversa'] = pd.to_datetime(df_silver['data_inversa'], errors='coerce')
df_silver['dia_semana'] = df_silver['data_inversa'].dt.day_name(locale='pt_BR')
df_silver.dtypes

### Padronizando textos

In [ ]:
def padronizar_colunas(df_silver):
    df_silver.columns = (
        df_silver.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df_silver

df_silver = padronizar_colunas(df_silver)

In [ ]:
df_silver['data_inversa'].count()

In [ ]:
df_silver.shape

In [ ]:
df_silver.to_parquet('../data/2-silver/silver_acidentes.parquet', index=False)